<a href="https://colab.research.google.com/github/Leonanda1013/DataLoverz/blob/main/Competition/DSCO/DSCO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import numpy as np
import pandas as pd
import math
from collections import Counter
from urllib.parse import urlparse

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from lightgbm import LGBMClassifier

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
dfTrain = pd.read_csv('/content/drive/MyDrive/DATASET/dsco (2)/train.csv')

In [8]:
dfTrain.head()

,URL,url_length,has_ip_address,dot_count,https_flag,url_entropy,token_count,subdomain_count,query_param_count,tld_length,path_length,has_hyphen_in_domain,number_of_digits,tld_popularity,suspicious_file_extension,domain_name_length,percentage_numeric_chars,ClassLabel
0,https://www.womensweekly.com.sg,31,0,3,1,3.461320,6.0,2,1,2,0.00000,0,0,0,0,3,0.000000,1.0
1,http://116.53.34.145:34075/i,28,1,3,0,3.645593,7.0,2,1,9,2.00000,0,15,0,0,2,53.571429,0.0
2,http://58.23.215.31:8765/wzoptup.exe,36,1,4,0,4.086049,8.0,2,1,7,11.44075,0,13,0,1,3,36.111111,0.0
3,https://www.dudpro.co.il,24,0,3,1,3.772055,6.0,2,1,2,0.00000,0,0,0,0,2,0.000000,1.0
4,http://117.201.113.115:53518/i,30,1,3,0,3.819549,11.2,2,1,9,2.00000,0,17,0,0,3,0.371737,0.0


In [9]:
dfTest = pd.read_csv('/content/drive/MyDrive/DATASET/dsco (2)/test.csv')

In [10]:
dfTest.head()

,ID,URL,url_length,has_ip_address,dot_count,https_flag,url_entropy,token_count,subdomain_count,query_param_count,tld_length,path_length,has_hyphen_in_domain,number_of_digits,tld_popularity,suspicious_file_extension,domain_name_length,percentage_numeric_chars
0,1,https://www.fastpost.com,24,0,2,1,3.522055,5.917567,2,1,3,0.000000,0,0,1,0,3,0.000000
1,2,http://proxy.amazonscouts.com/revada/66df1acad...,62,0,3,0,3.812500,8.000000,1,1,3,33.000000,0,7,1,1,0,7.194527
2,3,http://42.179.239.133:58058/bin.sh,34,1,4,0,3.695947,8.000000,2,1,9,6.673771,0,16,0,0,3,47.058824
3,4,https://www.saddhamma.org,25,0,2,1,3.703465,5.917567,1,1,3,0.000000,0,0,0,0,9,0.000000
4,5,https://www.changeip.com,24,0,2,1,3.723800,5.000000,1,1,3,0.000000,0,0,1,0,8,0.000000


In [12]:
dfTrain.duplicated().sum()
dfTest.duplicated().sum()

np.int64(0)

In [13]:
dfTrain.info()
dfTest.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80974 entries, 0 to 80973
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   URL                        80974 non-null  object 
 1   url_length                 80974 non-null  int64  
 2   has_ip_address             80974 non-null  int64  
 3   dot_count                  80974 non-null  int64  
 4   https_flag                 80974 non-null  int64  
 5   url_entropy                80974 non-null  float64
 6   token_count                80974 non-null  float64
 7   subdomain_count            80974 non-null  int64  
 8   query_param_count          80974 non-null  int64  
 9   tld_length                 80974 non-null  int64  
 10  path_length                80974 non-null  float64
 11  has_hyphen_in_domain       80974 non-null  int64  
 12  number_of_digits           80974 non-null  int64  
 13  tld_popularity             80974 non-null  int

In [14]:
dfTest['URL'] = dfTest['URL'].str.strip()
dfTrain['URL'] = dfTrain['URL'].str.strip()

In [25]:
# cari URL yang punya lebih dari 1 label unik
conflict_url = (
    dfTrain.groupby('URL')['ClassLabel']
      .nunique()
      .loc[lambda x: x > 1]
      .index
)
print (conflict_url)
dfTrain.shape
# drop semua baris dengan URL konflik
df_clean = dfTrain[~dfTrain['URL'].isin(conflict_url)].reset_index(drop=True)
df_clean.shape




Index(['http://112.93.200.30:43218/bin.sh',
       'http://115.58.158.131:37407/bin.sh',
       'http://117.255.97.217:39202/Mozi.m',
       'http://123.11.200.219:55499/bin.sh', 'http://123.11.200.219:55499/i',
       'http://31.148.168.117:47246/bin.sh', 'http://59.95.86.9:40506/i',
       'http://61.0.215.215:35269/bin.sh', 'http://hailcocks.ru/mips',
       'https://analytics.usa.gov/', 'https://nytimes.com',
       'https://www.census.gov/popclock/',
       'https://www.charitynavigator.org/'],
      dtype='object', name='URL')


(80946, 18)

In [20]:
dfTrain[dfTrain['URL'] == 'http://112.93.200.30:43218/bin.sh']

,URL,url_length,has_ip_address,dot_count,https_flag,url_entropy,token_count,subdomain_count,query_param_count,tld_length,path_length,has_hyphen_in_domain,number_of_digits,tld_popularity,suspicious_file_extension,domain_name_length,percentage_numeric_chars,ClassLabel
916,http://112.93.200.30:43218/bin.sh,33,1,4,0,3.899714,8.0,2,1,8,6.673771,0,15,0,0,3,45.454545,0.0
37946,http://112.93.200.30:43218/bin.sh,33,1,4,0,3.899714,8.0,2,1,8,7.000000,0,15,1,0,3,45.454545,1.0


In [26]:
SUSPICIOUS_KEYWORDS = [
    "secure", "login", "log-in", "signin", "sign-in", "verify",
    "update", "account", "paypal", "bank", "confirm", "service",
    "wallet", "support",
]

SAFE_HOST_KEYWORDS = [
    "stackoverflow.com", "upwork.com", "github.com", "gitlab.com",
    "bitbucket.org", "discordapp.com", "cdn.discordapp.com",
    ".gov", ".edu", ".ac.", "wikipedia.org", "nps.edu", "pc.gov.",
]

SHELL_TOKENS   = ["bin.sh", ".sh", "wget", "curl"]
MOZI_TOKENS    = ["mozi", "Mozi"]
SCANNER_TOKENS = ["gpon", "linksys", "zmap"]


def shannon_entropy(s: str) -> float:
    if not s:
        return 0.0
    counts = Counter(s)
    n = len(s)
    return -sum((c / n) * math.log2(c / n) for c in counts.values())


def count_suspicious_keywords(s: str) -> int:
    s_low = s.lower()
    return sum(k in s_low for k in SUSPICIOUS_KEYWORDS)


def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    url_series = df["URL"].astype(str)

    # replace 0 with 1 so no multiplication is done with 0
    url_len = df["url_length"].replace(0, 1)

    df["digit_ratio"] = url_series.str.count(r"\d") / url_len
    df["letter_ratio"] = url_series.str.count(r"[A-Za-z]") / url_len
    df["special_char_ratio"] = url_series.str.count(r"[^A-Za-z0-9]") / url_len
    df["uppercase_ratio"] = url_series.str.count(r"[A-Z]") / url_len
    df["lowercase_ratio"] = url_series.str.count(r"[a-z]") / url_len

    path_depth = []
    has_at_symbol = []
    has_hex_encoding = []
    domain_entropy = []
    path_entropy = []
    domain_token_count = []
    suspicious_kw_count = []
    safe_host_flag = []
    ip_and_suspicious_ext = []
    deep_path_and_ext = []

    has_ip_col = "has_ip_address" in df.columns
    susp_ext_col = "suspicious_file_extension" in df.columns
    path_len = df["path_length"] if "path_length" in df.columns else pd.Series(0, index=df.index)

    has_port = []
    has_shell_keyword = []
    has_mozi_keyword = []
    has_scanner_keyword = []

    for i, u in enumerate(url_series):
        try:
            parsed = urlparse(u)
        except Exception:
            parsed = urlparse("")

        domain = parsed.netloc or ""
        path = parsed.path or ""
        query = parsed.query or ""

        full_path = path + "?" + query if query else path

        path_depth.append(path.count("/") if path else 0)

        has_at_symbol.append(int("@" in u))
        has_hex_encoding.append(int("%" in u))

        domain_entropy.append(shannon_entropy(domain))
        path_entropy.append(shannon_entropy(path))

        tokens = []
        if domain:
            for part in domain.split("."):
                tokens.extend(part.split("-"))
        tokens = [t for t in tokens if t]
        domain_token_count.append(len(tokens))

        suspicious_kw_count.append(count_suspicious_keywords(u))

        d_lower = domain.lower()
        safe_flag = any(safe_key in d_lower for safe_key in SAFE_HOST_KEYWORDS)
        safe_host_flag.append(int(safe_flag))

        if has_ip_col and susp_ext_col:
            ip_and_suspicious_ext.append(
                int(df["has_ip_address"].iloc[i] == 1 and df["suspicious_file_extension"].iloc[i] == 1)
            )
        else:
            ip_and_suspicious_ext.append(0)

        if susp_ext_col:
            deep_path_and_ext.append(
                int(path_len.iloc[i] > 30 and df["suspicious_file_extension"].iloc[i] == 1)
            )
        else:
            deep_path_and_ext.append(0)

        # extra IP/malware tokens
        has_port.append(int(":" in domain))
        low_u = u.lower()
        has_shell_keyword.append(int(any(tok in low_u for tok in SHELL_TOKENS)))
        has_mozi_keyword.append(int(any(tok in low_u for tok in MOZI_TOKENS)))
        has_scanner_keyword.append(int(any(tok in low_u for tok in SCANNER_TOKENS)))

    df["path_depth"] = path_depth
    df["has_at_symbol"] = has_at_symbol
    df["has_hex_encoding"] = has_hex_encoding
    df["domain_entropy"] = domain_entropy
    df["path_entropy"] = path_entropy
    df["domain_token_count"] = domain_token_count
    df["suspicious_kw_count"] = suspicious_kw_count
    df["safe_host_flag"] = safe_host_flag
    df["ip_and_suspicious_ext"] = ip_and_suspicious_ext
    df["deep_path_and_ext"] = deep_path_and_ext

    df["has_port"] = has_port
    df["has_shell_keyword"] = has_shell_keyword
    df["has_mozi_keyword"] = has_mozi_keyword
    df["has_scanner_keyword"] = has_scanner_keyword

    return df


# apply FE
train_fe = add_engineered_features(df_clean)
test_fe  = add_engineered_features(dfTest)

print("Train with FE shape:", train_fe.shape)
print("Test with FE shape :", test_fe.shape)

Train with FE shape: (80946, 37)
Test with FE shape : (20244, 37)
